# Stereo-seq OMNI (FFPE flank skin melanoma): Lmna WT vs KO (Fig. 4E–G; Supplementary Fig. S11B)
Tyr-CreERT2;BrafCA;Ptenflox/flox mice with Lmna wt/wt (`FFPE_Skin_WT1`) or Lmna flox/flox (`FFPE_Skin_KO1`).
- **Supplementary Fig. S11B** spatial distribution of cell types
- **Supplementary Fig. S11B** zoom-ins near the skin surface
- **Fig. 4E** Lrrc15 expression map, zoom-ins, and mean Lrrc15 expression of the 10 nearest fibroblast bins of each melanoma bin
- **Fig. 4F** EMT-like signature (AUCell, Hallmark EMT)
- **Fig. 4G** selected signatures increased in Lmna KO melanoma bins (AUCell)

In [ ]:
import os
import re
import importlib.metadata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import decoupler as dc
from scipy import stats

from spatial_utils import *

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')
os.makedirs('./figures', exist_ok=True)

for package in ['stereopy', 'numpy', 'pandas', 'matplotlib', 'scanpy', 'anndata', 'squidpy', 'decoupler']:
    try:
        print(f"{package} version: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed")

# Annotated bin30 objects: <GEO library name>_bin30_annotated.h5ad
SPATIAL_DIR = '../data/stereoseq_annotated'
# Dict order KO -> WT and sample_list order WT -> KO reproduce the original notebook
# (the EMT bar plot takes group order from combined_adata).
adataL = load_ffpe(SPATIAL_DIR, ['FFPE_Skin_KO1', 'FFPE_Skin_WT1'])
sample_list = ['FFPE_Skin_WT1', 'FFPE_Skin_KO1']
sc.set_figure_params(dpi=300, color_map = 'OrRd', facecolor=None, transparent=True)

## Supplementary Fig. S11B — Cell type maps

In [ ]:
for sample in sorted(sample_list):
    sc.pl.spatial(adataL[sample], color = ['cell_type'], spot_size=50, title = sample, save=f'FFPE_Whole_Anno_{sample}.pdf')

## Supplementary Fig. S11B — Zoom-in near the skin surface

In [ ]:
# Zoom-in regions (near the arrows in the manuscript). The sections in the manuscript were rotated and the
# zoom-ins cropped by hand; the rotation angle and windows below approximate those panels.
ZOOM = {'FFPE_Skin_WT1': dict(angle=-40, xlim=(5200, 8800), ylim=(-14000, -12300)),
        'FFPE_Skin_KO1': dict(angle=82, xlim=(11800, 15400), ylim=(8800, 10500))}

for sample in sample_list:
    fig, ax = plot_spatial_zoom(adataL[sample], 'cell_type', legend_loc=None, **ZOOM[sample])
    fig.set_label(f'{sample}_zoomin')
    plt.show()

## Fig. 4E — Lrrc15 expression

In [ ]:
for sample in sorted(sample_list):
    sc.pl.spatial(adataL[sample], color = ['Lrrc15'], spot_size=50, vmax = 4, cmap = 'Blues', save = f'Lrrc15_expression_{sample}.pdf')#

In [ ]:
for sample in sample_list:
    fig, ax = plot_spatial_zoom(adataL[sample], 'Lrrc15', cmap='Blues', vmin=0, vmax=4, **ZOOM[sample])
    fig.set_label(f'{sample}_Lrrc15_zoomin')
    plt.show()

## Fig. 4E — Mean Lrrc15 expression of the 10 nearest fibroblast bins of each melanoma bin

In [ ]:
from scipy.spatial import KDTree

# For each melanoma bin, mean Lrrc15 (log-normalised) of its 10 nearest fibroblast bins
# (melanoma = cell_type_with_pigment == 'Melanoma', i.e. pigmented melanoma bins are not used; as in the original analysis)
neighbor_mean = {}
for sample in sample_list:
    adata = adataL[sample]
    coords = adata.obsm['spatial']
    tumor = np.where(adata.obs['cell_type_with_pigment'] == 'Melanoma')[0]
    fibro = np.where(adata.obs['cell_type_with_pigment'] == 'Fibroblast')[0]
    _, nn = KDTree(coords[fibro]).query(coords[tumor], k=10)
    lrrc15 = np.asarray(adata.raw[:, 'Lrrc15'].X.todense()).ravel()
    neighbor_mean[sample] = lrrc15[fibro[nn]].mean(axis=1)

wt, ko = neighbor_mean['FFPE_Skin_WT1'], neighbor_mean['FFPE_Skin_KO1']
t, p = stats.ttest_ind(wt, ko)
print(f'WT: {wt.mean():.6f} ± {stats.sem(wt):.6f} (n = {len(wt)});  KO: {ko.mean():.6f} ± {stats.sem(ko):.6f} (n = {len(ko)});  '
      f'two-tailed t-test p = {p:.3g}')

GENOTYPE = {'FFPE_Skin_WT1': r'$\it{Lmna}^{wt/wt}$', 'FFPE_Skin_KO1': r'$\it{Lmna}^{flox/flox}$'}
fig, ax = plt.subplots(figsize=(1.6, 3))
ax.bar([0, 1], [wt.mean(), ko.mean()], yerr=[stats.sem(wt), stats.sem(ko)], color=['#DCDCDC', '#3333FF'],
       edgecolor='black', linewidth=0.5, width=0.6, capsize=3)
y = max(wt.mean(), ko.mean()) * 1.08
ax.plot([0, 0, 1, 1], [y, y * 1.02, y * 1.02, y], color='black', linewidth=0.8)
ax.text(0.5, y * 1.03, significance_stars(p), ha='center')
ax.set_ylim(0, y * 1.15)
ax.set_xticks([0, 1], [GENOTYPE[s] for s in sample_list], rotation=45, ha='right')
ax.set_ylabel('Mean expression')
ax.set_title('Lrrc15', style='italic', pad=12)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(False)
fig.set_label('Lrrc15_nearest_fibroblast_mean')
plt.show()

## Fig. 4F — EMT-like signature

In [ ]:
# Mean AUCell score of melanoma bins per section (cell_type == 'Melanoma', pigmented melanoma included);
# the error bars (SEM over bins) are too small to be visible
emt = {s: adataL[s].obs.loc[adataL[s].obs['cell_type'] == 'Melanoma',
                            'aucell_HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION'].values for s in sample_list}
t, p = stats.ttest_ind(emt['FFPE_Skin_WT1'], emt['FFPE_Skin_KO1'])
for s in sample_list:
    print(f'{s}: {emt[s].mean():.4f} ± {stats.sem(emt[s]):.6f} (n = {len(emt[s])})')
print(f'two-tailed t-test p = {p:.3g}')

GENOTYPE = {'FFPE_Skin_WT1': r'$\it{Lmna}^{wt/wt}$', 'FFPE_Skin_KO1': r'$\it{Lmna}^{flox/flox}$'}
fig, ax = plot_hbar([emt[s].mean() for s in sample_list], [GENOTYPE[s] for s in sample_list],
                    color=['#DCDCDC', '#3333FF'], xerr=[stats.sem(emt[s]) for s in sample_list],
                    xlabel='Mean AUCell score', title='EMT-like signature', figsize=(4, 1.8), top_axis=False)
ax.set_xlim(0, 0.12)
ax.set_xticks(np.arange(0, 0.121, 0.02))
x = max(emt[s].mean() for s in sample_list) * 1.04
ax.plot([x, x], [-0.3, 1.3], color='black', linewidth=0.8)
ax.text(x * 1.01, 0.5, significance_stars(p), va='center', rotation=270)
fig.set_label('EMT-like_signature')
plt.show()

## Fig. 4G — Signatures higher in Lmna KO melanoma bins (final AUCell scores in the h5ad files)

In [ ]:
adata = sc.concat(adataL)

In [ ]:
Mel = adata[adata.obs['cell_type'] == 'Melanoma'].copy()

In [ ]:
tmp = pd.DataFrame(index = Mel.obs['sample'].unique())
for key in Mel.obs.columns[:]:
    if key.startswith('aucell_'):
        tmp[f'{key}'] = 0
        for sample in Mel.obs['sample'].unique():
            tmp.loc[sample, f'{key}'] = np.mean(Mel[Mel.obs['sample'] == sample].obs[f'{key}'])

In [ ]:
tmp

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu

In [ ]:
tmp3 = pd.DataFrame(index = tmp.columns[:])
tmp3['FC'] = 0
tmp3['Pval'] = 0
for key in tmp.columns[:]:
    tmp3.loc[key, 'log2FC'] = np.log2(tmp.loc['FFPE_Skin_KO1', f'{key}']/tmp.loc['FFPE_Skin_WT1', f'{key}'])
    tmp3.loc[key, 'Pval'] = ttest_ind(list(adataL['FFPE_Skin_WT1'].obs[f'{key}']), list(adataL['FFPE_Skin_KO1'].obs[f'{key}']))[1]

In [ ]:
tmp3[tmp3['Pval'] < 0.05].sort_values('log2FC', ascending=False).to_csv('./figures/FFPE_Melanoma_Term_DEG.csv')

In [ ]:
Mel_deg = pd.read_csv("./figures/FFPE_Melanoma_Term_DEG.csv")

## Fig. 4G — Selected signatures increased in Lmna KO melanoma bins
The manuscript panel was drawn from an earlier AUCell scoring of the FFPE sections (250205), whose melanoma-bin log2FC
(KO / WT) are given in `../data/Fig4G_plot_values.csv` and plotted below. In that panel, the 2nd and 3rd bars were labelled
RESOLUTION_OF_DNA_RECOMBINATION_INTERMEDIATES and PROTEIN_DNA_COVALENT_CROSS_LINKING_REPAIR, but their values are those of
DISPLACEMENT_OF_DNA_GLYCOSYLASE_BY_APEX1 and MEIOTIC_DNA_DOUBLE_STRAND_BREAK_FORMATION (column `label_in_manuscript`);
the plot uses the gene sets the values belong to. The table above (`FFPE_Melanoma_Term_DEG.csv`) is computed from the AUCell
scores stored in the h5ad files (final scoring, 250214), which differ slightly from the earlier scoring.

In [ ]:
fig4g = pd.read_csv('../data/Fig4G_plot_values.csv')
display(fig4g)
fig, ax = plot_hbar(fig4g['log2FC'].values, fig4g['label'].values, color='#3333FF', xlabel='Log2FC', figsize=(4, 3.5))
fig.set_label('Selected_signatures_log2FC')
plt.show()

## Fibroblast bins (exploratory, not shown as a figure panel)

In [ ]:
Fibro = adata[adata.obs['cell_type'] == 'Fibroblast'].copy()

In [ ]:
tmp = pd.DataFrame(index = Fibro.obs['sample'].unique())
for key in Fibro.obs.columns[:]:
    if key.startswith('aucell_'):
        tmp[f'{key}'] = 0
        for sample in Fibro.obs['sample'].unique():
            tmp.loc[sample, f'{key}'] = np.mean(Fibro[Fibro.obs['sample'] == sample].obs[f'{key}'])

In [ ]:
tmp3 = pd.DataFrame(index = tmp.columns[:])
tmp3['FC'] = 0
tmp3['Pval'] = 0
for key in tmp.columns[:]:
    tmp3.loc[key, 'log2FC'] = np.log2(tmp.loc['FFPE_Skin_KO1', f'{key}']/tmp.loc['FFPE_Skin_WT1', f'{key}'])
    tmp3.loc[key, 'Pval'] = ttest_ind(list(adataL['FFPE_Skin_WT1'].obs[f'{key}']), list(adataL['FFPE_Skin_KO1'].obs[f'{key}']))[1]

In [ ]:
tmp3[tmp3['Pval'] < 0.05].sort_values('log2FC', ascending=False).to_csv('./figures/FFPE_Fibroblast_Term_DEG.csv')

In [ ]:
fibro_top10 = tmp3[tmp3['Pval'] < 0.05].sort_values('log2FC', ascending = False).iloc[:10][:]

In [ ]:
fibro_top10 = fibro_top10.reset_index()

In [ ]:
sns.barplot(fibro_top10, y = 'index',x = 'log2FC')
plt.ylabel('')
plt.xlabel('log2FoldChange')
plt.savefig("./figures/FFPE_fibro_KO_Term_DEG_top10.pdf")